# PXR Ensemble Docking — gnina GPU (Phase 2) v2

Docks all 5,231 compounds (4,718 training + 513 test) against **3 PXR crystal structures**
using gnina's CNN scoring. Saves results to Google Drive.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Receptor PDBQT files in `My Drive/openadmet_pxr/receptors/`:
   - `2qd9_protein.pdbqt`, `6tfi_protein.pdbqt`, `6tfib_h12free_protein.pdbqt`, `4ny9_protein.pdbqt`
3. Data parquets in `My Drive/openadmet_pxr/data/` (from UniMol run)

**Expected runtime:** ~3-4h on T4 GPU

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if not result.stdout:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install gnina and dependencies ────────────────────────────────────
import subprocess

# gnina binary (GPU-accelerated docking with CNN scoring)
!wget -q https://github.com/gnina/gnina/releases/download/v1.3/gnina -O /usr/local/bin/gnina
!chmod +x /usr/local/bin/gnina

# obabel for reliable PDBQT conversion (fallback if meeko fails)
!apt-get install -qq openbabel
!pip install -q rdkit pyarrow pandas meeko

# Verify gnina works
r = subprocess.run(['/usr/local/bin/gnina', '--version'], capture_output=True, text=True)
gnina_ok = r.returncode == 0
print('gnina:', r.stdout.strip() or r.stderr.strip()[:100])
if not gnina_ok:
    raise RuntimeError('gnina failed to run — check binary download')

# Verify obabel
r2 = subprocess.run(['obabel', '--version'], capture_output=True, text=True)
print('obabel:', r2.stdout.strip()[:80] or r2.stderr.strip()[:80])

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR      = '/content/drive/MyDrive/openadmet_pxr/data'
RECEPTOR_DIR  = '/content/drive/MyDrive/openadmet_pxr/receptors'
DOCKING_OUT   = '/content/drive/MyDrive/openadmet_pxr/docking'
os.makedirs(DOCKING_OUT, exist_ok=True)

receptors = {
    '2qd9':         {'pdbqt': f'{RECEPTOR_DIR}/2qd9_protein.pdbqt',          'cx': -2.795,  'cy': -0.789,  'cz': 24.304},
    '6tfi':         {'pdbqt': f'{RECEPTOR_DIR}/6tfi_protein.pdbqt',          'cx': 28.123,  'cy': 35.379,  'cz': 26.656},
    '6tfi_h12free': {'pdbqt': f'{RECEPTOR_DIR}/6tfib_h12free_protein.pdbqt', 'cx': 28.123,  'cy': 35.379,  'cz': 26.656},
    '4ny9':         {'pdbqt': f'{RECEPTOR_DIR}/4ny9_protein.pdbqt',          'cx': 14.217,  'cy': -10.979, 'cz': 1.225},
}
for name, cfg in receptors.items():
    exists = os.path.exists(cfg['pdbqt'])
    size = os.path.getsize(cfg['pdbqt']) // 1024 if exists else 0
    print(f'{"✓" if exists else "✗"} {name}: {size} KB')
    if not exists:
        raise FileNotFoundError(f'Upload {os.path.basename(cfg["pdbqt"])} to {RECEPTOR_DIR}/')

In [ ]:
# ── Cell 4: Load compound SMILES ──────────────────────────────────────────────
import pandas as pd
import numpy as np

PRIMARY = {'openadmet', 'analog_set1', 'htchem', 'htchem_semi_pure'}
folds_df = pd.read_parquet(f'{DATA_DIR}/butina_folds.parquet')
if 'source' in folds_df.columns:
    folds_df = folds_df[folds_df['source'].isin(PRIMARY)]
pec50_col  = 'pec50_median' if 'pec50_median' in folds_df.columns else 'pec50'
folds_df   = folds_df[folds_df[pec50_col].notna()].reset_index(drop=True)
smiles_col = 'smiles_std' if 'smiles_std' in folds_df.columns else 'smiles'

test_df     = pd.read_parquet(f'{DATA_DIR}/openadmet_test_std.parquet')
test_smi_col = 'smiles_std' if 'smiles_std' in test_df.columns else 'smiles'

train_smiles = folds_df[smiles_col].tolist()
test_smiles  = test_df[test_smi_col].tolist()
test_ids     = test_df['compound_id'].tolist()

print(f'Train: {len(train_smiles)} | Test: {len(test_smiles)}')

In [ ]:
# ── Cell 5: 3D conformer generation → PDBQT ──────────────────────────────────
# Uses obabel (reliable) with RDKit 3D conformer as intermediate.
# Handles both meeko API versions and falls back to obabel if meeko fails.

import os, subprocess, tempfile
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem

LIGAND_DIR = Path('/content/ligands')
LIGAND_DIR.mkdir(exist_ok=True)

def smiles_to_pdbqt(smiles, compound_id, out_dir):
    """SMILES → ETKDGv3+MMFF94 conformer → PDBQT (obabel, no meeko dependency)."""
    out_path = out_dir / f'{compound_id}.pdbqt'
    if out_path.exists() and out_path.stat().st_size > 0:
        return str(out_path)
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        mol = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        if AllChem.EmbedMolecule(mol, params) != 0:
            if AllChem.EmbedMolecule(mol, AllChem.EmbedParameters()) != 0:
                return None
        AllChem.MMFFOptimizeMolecule(mol)

        # Write SDF, convert to PDBQT with obabel (-xh = merge nonpolar H)
        sdf_path = out_dir / f'{compound_id}.sdf'
        writer = Chem.SDWriter(str(sdf_path))
        writer.write(mol)
        writer.close()

        r = subprocess.run(
            ['obabel', str(sdf_path), '-O', str(out_path), '-xh', '--partialcharge', 'gasteiger'],
            capture_output=True, text=True, timeout=30
        )
        sdf_path.unlink(missing_ok=True)
        if out_path.exists() and out_path.stat().st_size > 0:
            return str(out_path)
    except Exception as e:
        pass
    return None

# ── Smoke test with 3 compounds before the main loop ──
test_smi_list = ['CC(=O)Oc1ccccc1C(=O)O', 'c1ccccc1', 'CCO']
test_ok = sum(1 for i, s in enumerate(test_smi_list) if smiles_to_pdbqt(s, f'smoke_{i}', LIGAND_DIR))
print(f'Smoke test: {test_ok}/3 PDBQT files generated')
if test_ok == 0:
    raise RuntimeError('PDBQT generation completely failed — check obabel installation')

# ── Generate all conformers ──
all_smiles = train_smiles + test_smiles
all_ids    = [f'train_{i}' for i in range(len(train_smiles))] + [f'test_{c}' for c in test_ids]

success, failed = 0, 0
for smi, cid in zip(all_smiles, all_ids):
    path = smiles_to_pdbqt(smi, cid, LIGAND_DIR)
    if path:
        success += 1
    else:
        failed += 1
    if (success + failed) % 500 == 0:
        print(f'  {success+failed}/{len(all_smiles)} | success={success} failed={failed}')

print(f'\nConformers done: {success} success, {failed} failed')
if success == 0:
    raise RuntimeError('No conformers generated — check obabel')

In [ ]:
# ── Cell 6: Smoke-test gnina on ONE compound before full run ──────────────────
from rdkit import Chem
import numpy as np

POSES_DIR = Path('/content/poses')
POSES_DIR.mkdir(exist_ok=True)

def dock_compound(ligand_pdbqt, receptor_cfg, receptor_name, compound_id):
    """Run gnina, return score dict. Raises on unexpected failures."""
    out_path = POSES_DIR / f'{compound_id}_{receptor_name}.sdf'
    log_path = POSES_DIR / f'{compound_id}_{receptor_name}.log'

    cmd = [
        '/usr/local/bin/gnina',
        '--receptor',       receptor_cfg['pdbqt'],
        '--ligand',         ligand_pdbqt,
        '--out',            str(out_path),
        '--log',            str(log_path),
        '--center_x',       str(receptor_cfg['cx']),
        '--center_y',       str(receptor_cfg['cy']),
        '--center_z',       str(receptor_cfg['cz']),
        '--size_x', '28', '--size_y', '28', '--size_z', '28',
        '--exhaustiveness', '8',
        '--num_modes',      '5',
        '--cnn_scoring',    'rescore',
    ]

    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    scores = {'vina_score_1': np.nan, 'cnn_score_1': np.nan, 'cnn_affinity_1': np.nan,
              'vina_score_mean': np.nan, 'n_poses': 0, 'success': False,
              'error': ''}

    if result.returncode != 0:
        scores['error'] = result.stderr[:200]
        log_path.unlink(missing_ok=True)
        return scores

    if out_path.exists() and out_path.stat().st_size > 0:
        try:
            suppl = Chem.SDMolSupplier(str(out_path), sanitize=False)
            vina_scores, cnn_scores, cnn_affs = [], [], []
            for mol in suppl:
                if mol is None: continue
                if mol.HasProp('minimizedAffinity'):
                    vina_scores.append(float(mol.GetProp('minimizedAffinity')))
                if mol.HasProp('CNNscore'):
                    cnn_scores.append(float(mol.GetProp('CNNscore')))
                if mol.HasProp('CNNaffinity'):
                    cnn_affs.append(float(mol.GetProp('CNNaffinity')))
            if vina_scores:
                scores.update({'vina_score_1': vina_scores[0], 'vina_score_mean': np.mean(vina_scores),
                                'n_poses': len(vina_scores), 'success': True})
            if cnn_scores:  scores['cnn_score_1']    = cnn_scores[0]
            if cnn_affs:    scores['cnn_affinity_1'] = cnn_affs[0]
        except Exception as e:
            scores['error'] = str(e)
        out_path.unlink(missing_ok=True)
    else:
        scores['error'] = 'empty output SDF'
    log_path.unlink(missing_ok=True)
    return scores

# ── Smoke test gnina with compound 0 against 2qd9 ──
first_pdbqt = str(LIGAND_DIR / 'train_0.pdbqt')
if not os.path.exists(first_pdbqt):
    raise RuntimeError('train_0.pdbqt missing — Cell 5 failed')

smoke = dock_compound(first_pdbqt, receptors['2qd9'], '2qd9', 'smoke_dock')
print(f'Gnina smoke test: success={smoke["success"]}, vina={smoke["vina_score_1"]:.2f}, cnn={smoke["cnn_score_1"]}')
if smoke['error']:
    print(f'  Error: {smoke["error"]}')
if not smoke['success']:
    raise RuntimeError('gnina smoke test failed — cannot proceed')
print('✓ gnina working, starting full docking run')

In [ ]:
# ── Cell 7: Run docking — ALL receptors, ALL compounds (~3-4h) ───────────────
import time

results = {}
receptor_names = list(receptors.keys())
total = len(all_ids)
t0 = time.time()
n_success_total = 0

for idx, cid in enumerate(all_ids):
    ligand_pdbqt = str(LIGAND_DIR / f'{cid}.pdbqt')
    if not os.path.exists(ligand_pdbqt):
        results[cid] = {r: {'success': False, 'vina_score_1': np.nan, 'cnn_score_1': np.nan,
                             'cnn_affinity_1': np.nan, 'vina_score_mean': np.nan, 'n_poses': 0}
                        for r in receptor_names}
        continue

    results[cid] = {}
    compound_succeeded = False
    for rname, rcfg in receptors.items():
        scores = dock_compound(ligand_pdbqt, rcfg, rname, cid)
        results[cid][rname] = scores
        if scores['success']:
            compound_succeeded = True
    if compound_succeeded:
        n_success_total += 1

    if (idx + 1) % 100 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (idx + 1) * (total - idx - 1)
        print(f'{idx+1}/{total} | {elapsed/60:.1f}m elapsed | ETA {eta/60:.1f}m | {n_success_total} docked')

print(f'\nDone: {n_success_total}/{total} compounds docked across {len(receptors)} receptors')

In [ ]:
# ── Cell 8: Assemble parquets and save to Drive ───────────────────────────────
import shutil

def build_df(keys, id_field):
    rows = []
    for key in keys:
        row = {id_field: key}
        r = results.get(key, {})
        for rname in receptor_names:
            s = r.get(rname, {})
            for metric in ['vina_score_1', 'vina_score_mean', 'cnn_score_1', 'cnn_affinity_1', 'n_poses']:
                row[f'{rname}_{metric}'] = s.get(metric, np.nan)
        rows.append(row)
    return pd.DataFrame(rows)

train_keys = [f'train_{i}' for i in range(len(train_smiles))]
test_keys  = [f'test_{c}' for c in test_ids]

df_train = build_df(train_keys, 'compound_idx')
df_test  = build_df(test_keys,  'compound_id')

# Replace string keys with integer index for train
df_train['compound_idx'] = range(len(df_train))
# Fix test compound_id — strip 'test_' prefix
df_test['compound_id'] = test_ids

train_path = '/content/train_docking_scores_v2.parquet'
test_path  = '/content/test_docking_scores_v2.parquet'
df_train.to_parquet(train_path, index=False)
df_test.to_parquet(test_path, index=False)

shutil.copy(train_path, f'{DOCKING_OUT}/train_docking_scores_v2.parquet')
shutil.copy(test_path,  f'{DOCKING_OUT}/test_docking_scores_v2.parquet')

print(f'Train: {df_train.shape} | Test: {df_test.shape}')
print(f'Saved to: {DOCKING_OUT}')

vina_coverage = df_train[[c for c in df_train.columns if 'vina_score_1' in c]].notna().mean()
cnn_coverage  = df_train[[c for c in df_train.columns if 'cnn_score_1' in c]].notna().mean()
print(f'\nVina coverage: {vina_coverage.round(3).to_dict()}')
print(f'CNN  coverage: {cnn_coverage.round(3).to_dict()}')